# Fine-tune Qwen2.5-3B-Instruct bằng Unsloth (DoRA) trên VlogQA
Notebook này hướng dẫn fine-tune mô hình **Qwen2.5-3B-Instruct** với **DoRA (Weight-Decomposed Low-Rank Adaptation)** qua thư viện **Unsloth** cho tập dữ liệu **VlogQA** (đọc hiểu hội thoại tiếng Việt).

VlogQA là dataset **Extractive QA** (dạng SQuAD): mô hình cần trích xuất một đoạn văn bản ngắn từ context làm câu trả lời.

**DoRA vs LoRA:** DoRA phân tách weight thành magnitude và direction, fine-tune cả hai thành phần. Kỹ thuật này thường cho kết quả tốt hơn LoRA trên các tác vụ chuyên biệt (chỉ cần bật `use_dora=True`).

**Evaluation metrics:** Exact Match (EM) và F1-score.

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ để tránh xung đột
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
# Bước 2: Cài đặt lại PyTorch với CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# Bước 3: Cài đặt transformers, trl, peft, accelerate, bitsandbytes, xformers và unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo

## Bước 4: Load Model với FastLanguageModel

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192  # Tăng từ 1536 → bao phủ P90 VlogQA thực tế (~7052 tokens)
dtype = None           # None để auto-detect (BFloat16 cho 5070 Ti - Blackwell arch)
load_in_4bit = False   # DoRA thuần: nạp model BF16 (~6GB), không quantize

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Tải model thành công!")

## Bước 5: Cấu hình DoRA Adapter

**Điểm khác biệt so với LoRA:** Chỉ cần đặt `use_dora=True`.

DoRA phân tách weight W thành:
- **Magnitude** (m): vector chuẩn hóa cột
- **Direction** (V): ma trận hướng được chuẩn hóa theo cột

Sau đó áp dụng LoRA trên phần Direction, giữ Magnitude có thể huấn luyện riêng biệt → kết quả thường tốt hơn LoRA thuần.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # Rank của DoRA (16 là cân bằng tốt)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,                 # Scaling factor (thường = 2 * r)
    lora_dropout = 0.05,             # Dropout nhẹ để tránh overfitting
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Tiết kiệm VRAM 30%
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
    # ============================================================
    # DoRA: Bật Weight-Decomposed Low-Rank Adaptation
    # Phân tách weight = magnitude * (direction + LoRA)
    # ============================================================
    use_dora = True,
)

print("Cấu hình DoRA thành công!")
print(f"Số tham số có thể huấn luyện: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Bước 6: Load và Tiền xử lý Dữ liệu VlogQA

In [ ]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                # Lấy câu trả lời đầu tiên (có thể có nhiều annotators)
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""  # Bỏ qua câu không có đáp án
                if answer:  # Chỉ lấy mẫu có đáp án
                    samples.append({
                        "context":  context,
                        "question": question,
                        "answer":   answer,
                        "id":       qa.get("id", ""),
                    })
    return samples


# ==========================================
# Anchor-based Context Truncation
# Cắt context thông minh, ưu tiên giữ answer span
# ==========================================
MAX_CONTEXT_TOKENS = 7500  # Ngân sách context trong cửa sổ 8192 tokens

def truncate_context_around_answer(context, answer, tokenizer, max_ctx_tokens=MAX_CONTEXT_TOKENS):
    """
    Cắt context về max_ctx_tokens tokens, ưu tiên giữ đoạn chứa answer span.
    Đảm bảo answer luôn nằm trong context sau khi cắt.
    """
    answer_start = context.find(answer)
    if answer_start == -1:
        # Không tìm được answer → cắt từ đầu
        ctx_ids = tokenizer.encode(context, add_special_tokens=False)
        if len(ctx_ids) <= max_ctx_tokens:
            return context
        return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)

    # Chia context thành 3 phần: trước / answer / sau
    before_text = context[:answer_start]
    after_text  = context[answer_start + len(answer):]

    before_ids = tokenizer.encode(before_text, add_special_tokens=False)
    answer_ids = tokenizer.encode(answer,       add_special_tokens=False)
    after_ids  = tokenizer.encode(after_text,   add_special_tokens=False)

    # Nếu tổng đã trong ngưỡng → giữ nguyên
    if len(before_ids) + len(answer_ids) + len(after_ids) <= max_ctx_tokens:
        return context

    # Phân bổ budget cho 2 bên
    budget = max_ctx_tokens - len(answer_ids)
    half   = budget // 2

    before_keep = before_ids[-half:]          # Giữ phần cuối của đoạn "trước"
    after_keep  = after_ids[:budget - half]   # Giữ phần đầu của đoạn "sau"

    merged_ids = before_keep + answer_ids + after_keep
    return tokenizer.decode(merged_ids, skip_special_tokens=True)


# ==========================================
# Prompt template cho Extractive QA
# ==========================================
SYSTEM_PROMPT = (
    "Bạn là công cụ trích xuất thông tin. "
    "Nhiệm vụ: tìm và COPY NGUYÊN VẸN (từng ký tự) một đoạn ngắn từ đoạn văn được cung cấp để trả lời câu hỏi. "
    "TUYỆT ĐỐI không paraphrase, không thêm bớt từ, không giải thích. "
    "Chỉ trả về đúng chuỗi ký tự xuất hiện trong đoạn văn."
)

def format_prompt_train(context, question, answer, tokenizer):
    """Tạo prompt đầy đủ (cả input lẫn output) để huấn luyện."""
    # Áp dụng anchor truncation trước khi tạo prompt
    context_cropped = truncate_context_around_answer(context, answer, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "
                f"đúng cụm từ xuất hiện trong đoạn văn.\n\n"
                f"Đoạn văn:\n{context_cropped}\n\n"
                f"Câu hỏi: {question}\n\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
        {
            "role": "assistant",
            "content": answer,
        }
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return text


def format_prompt_inference(context, question, tokenizer):
    """Tạo prompt cho inference (không có đáp án)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "
                f"đúng cụm từ xuất hiện trong đoạn văn.\n\n"
                f"Đoạn văn:\n{context}\n\n"
                f"Câu hỏi: {question}\n\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text


# Đường dẫn dữ liệu (điều chỉnh nếu cần)
TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"
TEST_PATH  = "test.json"

train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")
print(f"Ví dụ mẫu đầu tiên:")
print(f"  Context (100 ký tự đầu): {train_samples[0]['context'][:100]}")
print(f"  Question: {train_samples[0]['question']}")
print(f"  Answer: {train_samples[0]['answer']}")

In [ ]:
# ==========================================
# Tạo Dataset HuggingFace từ mẫu train
# ==========================================
texts = [
    format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
    for s in train_samples
]

dataset = Dataset.from_dict({"text": texts})
print(f"Dataset đã tạo: {len(dataset)} mẫu")
print(f"\nVí dụ prompt (200 ký tự đầu):\n{texts[0][:200]}")

## Bước 7: Cấu hình và Chạy SFT Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import sys

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,          # 8192
    dataset_num_proc = 1,      # Dùng 1 process để tránh lỗi pickle
    packing = True,            # Tận dụng GPU: pack nhiều mẫu ngắn thành 1 sequence 8192
    args = TrainingArguments(
        per_device_train_batch_size = 2,      # Tăng từ 2 → 4 (15GB VRAM dư sức)
        gradient_accumulation_steps = 8,      # Effective batch = 2 * 8 = 16
        warmup_steps = 50,                    # Tăng từ 10 → 50 (ổn định với batch lớn hơn)
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),       # 5070 Ti hỗ trợ BFloat16
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_vlogqa_dora",  # Thư mục riêng cho DoRA
        save_strategy = "epoch",
        report_to = "none",
    ),
)

# Fix sys.modules để tránh lỗi pickle với unsloth
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

# Bắt đầu train!
print("\nBắt đầu fine-tuning với DoRA...")
trainer_stats = trainer.train()

# Chỉ lưu DoRA weights (nhẹ, vài chục MB)
DORA_SAVE_PATH = "qwen2.5-3b-instruct-dora-vlogqa"

model.save_pretrained(DORA_SAVE_PATH)
tokenizer.save_pretrained(DORA_SAVE_PATH)

print(f"Đã lưu thành công mô hình DoRA tại: {DORA_SAVE_PATH}")

# Tùy chọn: Merge DoRA vào model gốc và lưu dạng full weights
# model.save_pretrained_merged("qwen2.5-3b-instruct-vlogqa-dora-merged", tokenizer, save_method="merged_16bit")

## Bước 8: Inference nhanh để kiểm tra

In [ ]:
# Kích hoạt chế độ sinh token nhanh của Unsloth (2x tốc độ)
# FastLanguageModel.for_inference(model) # Tạm tắt do bug của Unsloth với DoRA

# Thử nghiệm với mẫu đầu tiên trong tập train
sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
# Đảm bảo model đang ở chế độ evaluation
model.eval()

outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 64,
    use_cache = True,
    do_sample = False,    # Greedy decoding cho QA
    temperature = 1.0,
)

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE ---")
print(f"Câu hỏi : {sample['question']}")
print(f"Đáp án đúng : {sample['answer']}")
print(f"Model trả lời: {response}")



## Bước 10: Đánh giá trên tập Test

In [ ]:
import json
import re
import string
import unicodedata
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from collections import Counter

# ==========================================
# CẤU HÌNH
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH    = "qwen2.5-3b-instruct-dora-vlogqa"  # Đường dẫn DoRA đã lưu
TEST_DATA_PATH  = "test.json"
MAX_EVAL_TOKENS = 7500   # Ngân sách context khi inference (giống training)
SYSTEM_PROMPT = (
    "Bạn là công cụ trích xuất thông tin. "
    "Nhiệm vụ: tìm và COPY NGUYÊN VẸN (từng ký tự) một đoạn ngắn từ đoạn văn được cung cấp để trả lời câu hỏi. "
    "TUYỆT ĐỐI không paraphrase, không thêm bớt từ, không giải thích. "
    "Chỉ trả về đúng chuỗi ký tự xuất hiện trong đoạn văn."
)

In [ ]:
# ==========================================
# LOAD MODEL ĐÃ FINE-TUNE (TỐI ƯU TỐC ĐỘ)
# ==========================================
from unsloth import FastLanguageModel
import torch

print("Loading Tokenizer và Model bằng Unsloth FastInference...")
model_eval, tokenizer_eval = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_PATH,  # Load thẳng folder DoRA, Unsloth tự tìm base model
    max_seq_length = 8192,
    dtype = torch.bfloat16,     # BF16 tối ưu cho RTX 40-series
    load_in_4bit = False,
)

# KÍCH HOẠT NATIVE FAST INFERENCE CỦA UNSLOTH (Tăng 2x tốc độ sinh token)
# Tạm tắt FastInference
model_eval.eval()
print("Đang merge DoRA weights (Eval)...")
from peft import PeftModel
if isinstance(model_eval, PeftModel):
    model_eval = model_eval.merge_and_unload()
print("Merge hoàn tất!")

print("Load model hoàn tất! (Sử dụng Unsloth FastInference + FlashAttention)")


In [ ]:
# ==========================================
# HÀM TIỆN ÍCH: CHUẨN HÓA VĂN BẢN
# ==========================================
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())
    return text


def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))


def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 and len(true_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)


def get_best_score(prediction: str, answers: list) -> tuple:
    em_scores = [compute_exact_match(prediction, ans["text"]) for ans in answers]
    f1_scores = [compute_f1_token(prediction, ans["text"]) for ans in answers]
    return max(em_scores), max(f1_scores)


# ==========================================
# Best-Span Post-processing
# Tìm substring trong context khớp nhất với prediction của model
# ==========================================
def find_best_span(prediction: str, context: str) -> str:
    """
    Sau khi model sinh ra text, tìm substring trong context
    có token-F1 cao nhất so với prediction.
    Rất hiệu quả khi model paraphrase thay vì copy nguyên văn.
    """
    pred_norm = normalize_text(prediction)
    if not pred_norm:
        return prediction

    pred_words = pred_norm.split()
    ctx_norm   = normalize_text(context)
    ctx_words  = ctx_norm.split()
    ctx_orig   = context.split()   # giữ nguyên văn để reconstruct

    n = len(pred_words)
    if n == 0 or len(ctx_words) == 0:
        return prediction

    # Tính F1 của raw prediction với context-normalized để làm baseline
    baseline_f1 = compute_f1_token(pred_norm, ctx_norm)
    best_f1     = baseline_f1
    best_span   = prediction

    # Slide window kích thước n//2 đến 2n+5 qua context
    min_w = max(1, n // 2)
    max_w = min(2 * n + 5, len(ctx_words))

    for w in range(min_w, max_w + 1):
        for i in range(len(ctx_words) - w + 1):
            span_norm  = " ".join(ctx_words[i:i + w])
            f1 = compute_f1_token(pred_norm, span_norm)
            if f1 > best_f1:
                best_f1  = f1
                best_span = " ".join(ctx_orig[i:i + w])

    return best_span


# ==========================================
# HÀM TẠO PROMPT INFERENCE (có pre-truncate context)
# ==========================================
def truncate_context_for_eval(context, tokenizer, max_ctx_tokens=MAX_EVAL_TOKENS):
    """
    Pre-truncate context trước khi đưa vào prompt,
    đảm bảo question LUÔN xuất hiện trong prompt.
    """
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    if len(ctx_ids) <= max_ctx_tokens:
        return context
    return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)


def format_prompt_inference_eval(context, question, tokenizer):
    """Tạo prompt cho inference với pre-truncated context."""
    context_cropped = truncate_context_for_eval(context, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đoạn văn:\n{context_cropped}\n\n"
                f"Câu hỏi: {question}\n\n"
                f"Trích xuất nguyên văn từ đoạn văn (không thêm bớt):"
            )
        },
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],   # Giữ toàn bộ danh sách đáp án
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")

In [ ]:
# ==========================================
# CHẠY INFERENCE TRÊN TOÀN BỘ TẬP TEST
# ==========================================
all_em_raw      = []   # EM không post-process
all_f1_raw      = []   # F1 không post-process
all_em_span     = []   # EM sau best-span matching
all_f1_span     = []   # F1 sau best-span matching
all_predictions = []

print(f"\nBắt đầu đánh giá trên {len(test_samples)} câu hỏi...\n")

for sample in tqdm(test_samples, desc="Evaluating"):
    prompt = format_prompt_inference_eval(
        context   = sample["context"],
        question  = sample["question"],
        tokenizer = tokenizer_eval
    )

    # Tokenize (context đã được pre-truncate trong format_prompt_inference_eval)
    inputs = tokenizer_eval(
        prompt,
        return_tensors = "pt",
        truncation = True,
        max_length = 8192,
    ).to(model_eval.device)

    # Generate
    with torch.no_grad():
        outputs = model_eval.generate(
            **inputs,
            max_new_tokens = 64,           # tăng từ 50 → 64 cho an toàn
            do_sample = False,
            temperature = 1.0,
            pad_token_id = tokenizer_eval.pad_token_id,
            eos_token_id = tokenizer_eval.eos_token_id,
        )

    # Decode raw prediction
    input_length   = inputs["input_ids"].shape[1]
    generated_ids  = outputs[0][input_length:]
    raw_prediction = tokenizer_eval.decode(generated_ids, skip_special_tokens=True).strip()
    raw_prediction = raw_prediction.split("\n")[0].strip()

    # Post-processing: snap về span trong context gốc
    span_prediction = find_best_span(raw_prediction, sample["context"])

    # Tính metric cho cả raw và span
    em_raw,  f1_raw  = get_best_score(raw_prediction,  sample["answers"])
    em_span, f1_span = get_best_score(span_prediction, sample["answers"])

    all_em_raw.append(em_raw);   all_f1_raw.append(f1_raw)
    all_em_span.append(em_span); all_f1_span.append(f1_span)
    all_predictions.append({
        "id":             sample["id"],
        "question":       sample["question"],
        "ground_truth":   sample["answers"][0]["text"],
        "raw_prediction":  raw_prediction,
        "span_prediction": span_prediction,
        "em_raw":  em_raw,  "f1_raw":  f1_raw,
        "em_span": em_span, "f1_span": f1_span,
    })

    tqdm.write(
        f"[{len(all_predictions)}/{len(test_samples)}] "
        f"Q: {sample['question'][:40]}... | "
        f"Truth: {sample['answers'][0]['text'][:30]} | "
        f"Raw: {raw_prediction[:30]} | "
        f"Span: {span_prediction[:30]} | "
        f"EM_raw={em_raw} EM_span={em_span} "
        f"F1_raw={f1_raw:.3f} F1_span={f1_span:.3f}"
    )

print("\nHoàn tất inference!")

In [ ]:
# ==========================================
# IN KẾT QUẢ ĐÁNH GIÁ
# ==========================================
total = len(all_em_raw)

em_raw_score   = sum(all_em_raw)  / total * 100
f1_raw_score   = sum(all_f1_raw)  / total * 100
em_span_score  = sum(all_em_span) / total * 100
f1_span_score  = sum(all_f1_span) / total * 100

print("\n" + "=" * 60)
print("   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA")
print("   Model: Qwen2.5-3B-Instruct + DoRA (Unsloth)")
print("=" * 60)
print(f"  Tổng số câu hỏi test: {total}")
print()
print(f"  {'Metric':<25} {'Raw (no post-proc)':>20} {'Best-Span':>20}")
print(f"  {'-'*65}")
print(f"  {'Exact Match (EM)':<25} {em_raw_score:>19.2f}% {em_span_score:>19.2f}%")
print(f"  {'F1-score (token)':<25} {f1_raw_score:>19.2f}% {f1_span_score:>19.2f}%")
print("=" * 60)

# Phân phối F1 (span version)
f1_buckets = {"0-0.2": 0, "0.2-0.5": 0, "0.5-0.8": 0, "0.8-1.0": 0, "1.0": 0}
for f1 in all_f1_span:
    if f1 == 1.0:    f1_buckets["1.0"] += 1
    elif f1 >= 0.8:  f1_buckets["0.8-1.0"] += 1
    elif f1 >= 0.5:  f1_buckets["0.5-0.8"] += 1
    elif f1 >= 0.2:  f1_buckets["0.2-0.5"] += 1
    else:            f1_buckets["0-0.2"] += 1

print(f"\n  EM đúng (span): {sum(all_em_span)}/{total} câu")
print("\n  Phân phối F1-score (Best-Span):")
for bucket, count in f1_buckets.items():
    print(f"    F1 = {bucket}: {count} câu ({count/total*100:.1f}%)")

# Lưu kết quả
with open("eval_results_vlogqa_3b_dora.json", "w", encoding="utf-8") as f:
    json.dump({
        "model":   "Qwen2.5-3B-Instruct + DoRA",
        "adapter": ADAPTER_PATH,
        "em_raw":    round(em_raw_score,  4),
        "f1_raw":    round(f1_raw_score,  4),
        "em_span":   round(em_span_score, 4),
        "f1_span":   round(f1_span_score, 4),
        "total":     total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa_3b_dora.json")

## Bước Bổ Sung: Load lại Model Độc Lập và Test nhanh 1 sample trong test.json

In [ ]:
import torch
from unsloth import FastLanguageModel
import json

# ==============================================================
# 1. LOAD MODEL TỪ ĐĨA (Base Model + DoRA Adapter)
# ==============================================================
print("Loading Tokenizer và Model từ thư mục đã lưu...")
model_test, tokenizer_test = FastLanguageModel.from_pretrained(
    model_name = "qwen2.5-3b-instruct-dora-vlogqa",  # Thư mục lưu adapter của bạn
    max_seq_length = 8192,
    dtype = torch.bfloat16,
    load_in_4bit = False,
)

# KHÔNG DÙNG FastLanguageModel.for_inference() vì lỗi tương thích DoRA
model_test.eval()

# BẮT BUỘC VỚI DORA: Merge weights vào base model trước khi generate để tránh bug KV Cache của PEFT
print("Đang merge DoRA weights vào Base model...")
from peft import PeftModel
if isinstance(model_test, PeftModel):
    model_test = model_test.merge_and_unload()
print("Merge hoàn tất!")
print("Load model hoàn tất!")

# ==============================================================
# 2. ĐỌC SAMPLE ĐẦU TIÊN TỪ FILE TEST
# ==============================================================
first_sample = None
with open("test.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)
    
for item in raw_data:
    for paragraph in item["paragraphs"]:
        for qa in paragraph["qas"]:
            if qa["answers"]:
                first_sample = {
                    "context": paragraph["context"],
                    "question": qa["question"],
                    "answer": qa["answers"][0]["text"]
                }
                break
        if first_sample: break
    if first_sample: break

print(f"\nĐang test câu hỏi: {first_sample['question']}")

# ==============================================================
# 3. TẠO PROMPT (Dùng chuẩn Chat Template)
# ==============================================================
SYSTEM_PROMPT = (
    "Bạn là công cụ trích xuất thông tin. "
    "Nhiệm vụ: tìm và COPY NGUYÊN VẸN (từng ký tự) một đoạn ngắn từ đoạn văn được cung cấp để trả lời câu hỏi. "
    "TUYỆT ĐỐI không paraphrase, không thêm bớt từ, không giải thích. "
    "Chỉ trả về đúng chuỗi ký tự xuất hiện trong đoạn văn."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            f"Đoạn văn:\n{first_sample['context'][:7000]}\n\n"
            f"Câu hỏi: {first_sample['question']}\n\n"
            f"Trích xuất nguyên văn từ đoạn văn (không thêm bớt):"
        )
    },
]
prompt = tokenizer_test.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ==============================================================
# 4. CHẠY INFERENCE VÀ IN KẾT QUẢ
# ==============================================================
inputs = tokenizer_test(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model_test.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_new_tokens = 64,
        do_sample = False,
        pad_token_id = tokenizer_test.pad_token_id,
        eos_token_id = tokenizer_test.eos_token_id,
    )

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer_test.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE ĐỘC LẬP (TEST) ---")
print(f"Câu hỏi       : {first_sample['question']}")
print(f"Đáp án đúng   : {first_sample['answer']}")
print(f"Model trả lời : {response}")


## Bước 11: (Tùy chọn) Đánh giá Siêu Tốc với vLLM

`vLLM` là một thư viện phục vụ inference cực nhanh. Thay vì đánh giá từng câu mất hàng giờ, `vLLM` sẽ nhóm các câu hỏi lại và chạy song song, chỉ mất vài phút để hoàn thành file test.

**LƯU Ý:** vLLM yêu cầu mô hình phải được gộp (merge) sẵn thành một Base Model hoàn chỉnh trên đĩa, không đọc trực tiếp Adapter. Do đó chúng ta cần chạy lệnh xuất mô hình trước.

In [ ]:
!pip install vllm

import torch
from unsloth import FastLanguageModel

print("Đang nạp mô hình và DoRA adapter từ đĩa...")
model_export, tokenizer_export = FastLanguageModel.from_pretrained(
    model_name = "qwen2.5-3b-instruct-dora-vlogqa",
    max_seq_length = 8192,
    dtype = torch.bfloat16,
    load_in_4bit = False,
)

MERGED_DIR = "qwen2.5-3b-vlogqa-merged-full"
print(f"Đang gộp DoRA và lưu vĩnh viễn ra thư mục: {MERGED_DIR}...")
model_export.save_pretrained_merged(MERGED_DIR, tokenizer_export, save_method="merged_16bit")
print("Hoàn tất xuất mô hình!")


---
### ⚠️ YÊU CẦU BẮT BUỘC TRƯỚC KHI CHẠY CELL TIẾP THEO ⚠️
Bạn **PHẢI RESTART KERNEL** (Vào menu Kernel -> Restart Kernel) để giải phóng toàn bộ VRAM của PyTorch.
Nếu không Restart, vLLM sẽ báo lỗi `CUDA Out of Memory` vì VRAM đã bị PyTorch chiếm giữ.
Sau khi Restart Kernel, bạn kéo thẳng xuống chạy Cell dưới đây (Không cần chạy lại các cell phía trên).

In [ ]:
import json
import time
from vllm import LLM, SamplingParams
import unicodedata
import string
from collections import Counter

# 1. Cấu hình vLLM
MODEL_PATH = "qwen2.5-3b-vlogqa-merged-full"
TEST_DATA_PATH = "test.json"
MAX_MODEL_LEN = 8192

print(f"Khởi tạo vLLM engine với model: {MODEL_PATH}")
llm = LLM(
    model=MODEL_PATH,
    max_model_len=MAX_MODEL_LEN,
    trust_remote_code=True,
    tensor_parallel_size=1, # Nếu có nhiều GPU, tăng số này lên
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=64,
)

# 2. Đọc dữ liệu
def load_test_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": paragraph["context"],
                        "question": qa["question"],
                        "answers": qa["answers"],
                    })
    return samples

test_samples = load_test_data(TEST_DATA_PATH)
print(f"Tổng số câu hỏi: {len(test_samples)}")

SYSTEM_PROMPT = (
    "Bạn là công cụ trích xuất thông tin. "
    "Nhiệm vụ: tìm và COPY NGUYÊN VẸN (từng ký tự) một đoạn ngắn từ đoạn văn được cung cấp để trả lời câu hỏi. "
    "TUYỆT ĐỐI không paraphrase, không thêm bớt từ, không giải thích. "
    "Chỉ trả về đúng chuỗi ký tự xuất hiện trong đoạn văn."
)

prompts = []
for sample in test_samples:
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nĐoạn văn:\n{sample['context'][:7000]}\n\nCâu hỏi: {sample['question']}\n\nTrích xuất nguyên văn từ đoạn văn (không thêm bớt):<|im_end|>\n<|im_start|>assistant\n"
    prompts.append(prompt)

# 3. Chạy Inference Siêu tốc
print("Bắt đầu sinh token hàng loạt (Batch Generation)...")
start_time = time.time()
outputs = llm.generate(prompts, sampling_params)
end_time = time.time()
print(f"Hoàn tất Inference trong {end_time - start_time:.2f} giây!")

# 4. Tính toán Metric
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def compute_exact_match(prediction, truth):
    return int(normalize_text(prediction) == normalize_text(truth))

def compute_f1(prediction, truth):
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(truth).split()
    if not pred_tokens and not true_tokens: return 1.0
    if not pred_tokens or not true_tokens: return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0: return 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)

all_predictions = []
em_scores, f1_scores = [], []

for i, output in enumerate(outputs):
    sample = test_samples[i]
    prediction = output.outputs[0].text.strip()
    
    em = max(compute_exact_match(prediction, ans["text"]) for ans in sample["answers"])
    f1 = max(compute_f1(prediction, ans["text"]) for ans in sample["answers"])
    
    em_scores.append(em)
    f1_scores.append(f1)
    
    all_predictions.append({
        "id": sample["id"],
        "question": sample["question"],
        "truth": sample["answers"][0]["text"],
        "prediction": prediction,
        "em": em,
        "f1": f1
    })

final_em = sum(em_scores) / len(em_scores) * 100
final_f1 = sum(f1_scores) / len(f1_scores) * 100

print(f"\nKẾT QUẢ CUỐI CÙNG:")
print(f"Exact Match (EM) : {final_em:.2f}%")
print(f"F1-score         : {final_f1:.2f}%")

with open("eval_results_vllm.json", "w", encoding="utf-8") as f:
    json.dump({
        "em": final_em, "f1": final_f1,
        "time_seconds": end_time - start_time,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)
print("Đã lưu kết quả chi tiết vào eval_results_vllm.json")


## Bước 12: Đánh giá Nhanh với HuggingFace Native Batching

Vì `vLLM` hay bị lỗi xung đột phiên bản CUDA trên một số Server, đây là giải pháp thay thế hoàn hảo.
Code này sử dụng bộ thư viện PyTorch/Transformers có sẵn của bạn nhưng tăng `Batch Size = 8` để chạy nhanh gấp 8 lần bình thường mà không cần cài thêm gì cả.

In [ ]:
import json
import time
import torch
import unicodedata
import string
from collections import Counter
from tqdm.auto import tqdm
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. LOAD MODEL (Chạy Native PyTorch)
print("Đang nạp mô hình...")
MODEL_PATH = "qwen2.5-3b-vlogqa-merged-full"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.padding_side = 'left'  # Bắt buộc padding bên trái cho quá trình Generate

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)
model.eval()

# 2. CHUẨN BỊ DỮ LIỆU
with open("test.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)
    
test_samples = []
for item in raw_data:
    for paragraph in item["paragraphs"]:
        for qa in paragraph["qas"]:
            if qa["answers"]:
                test_samples.append({
                    "id": qa.get("id", ""),
                    "context": paragraph["context"],
                    "question": qa["question"],
                    "answers": qa["answers"],
                })

SYSTEM_PROMPT = (
    "Bạn là công cụ trích xuất thông tin. "
    "Nhiệm vụ: tìm và COPY NGUYÊN VẸN (từng ký tự) một đoạn ngắn từ đoạn văn được cung cấp để trả lời câu hỏi. "
    "TUYỆT ĐỐI không paraphrase, không thêm bớt từ, không giải thích. "
    "Chỉ trả về đúng chuỗi ký tự xuất hiện trong đoạn văn."
)

prompts = []
for sample in test_samples:
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nĐoạn văn:\n{sample['context'][:7000]}\n\nCâu hỏi: {sample['question']}\n\nTrích xuất nguyên văn từ đoạn văn (không thêm bớt):<|im_end|>\n<|im_start|>assistant\n"
    prompts.append(prompt)

# 3. CHẠY INFERENCE THEO BATCH (BATCH SIZE = 8)
BATCH_SIZE = 8
all_predictions = []
print(f"Bắt đầu đánh giá với Batch Size = {BATCH_SIZE}...")
start_time = time.time()

for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
    batch_prompts = prompts[i:i+BATCH_SIZE]
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=8192).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        
    for j, output in enumerate(outputs):
        input_len = inputs['input_ids'][j].shape[0]
        pred_tokens = output[input_len:]
        pred_text = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip()
        all_predictions.append(pred_text)

end_time = time.time()
print(f"Hoàn tất trong {end_time - start_time:.2f} giây!")

# 4. TÍNH TOÁN METRIC
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def compute_exact_match(prediction, truth):
    return int(normalize_text(prediction) == normalize_text(truth))

def compute_f1(prediction, truth):
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(truth).split()
    if not pred_tokens and not true_tokens: return 1.0
    if not pred_tokens or not true_tokens: return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0: return 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)

em_scores, f1_scores = [], []
results_json = []

for idx, pred in enumerate(all_predictions):
    sample = test_samples[idx]
    em = max(compute_exact_match(pred, ans["text"]) for ans in sample["answers"])
    f1 = max(compute_f1(pred, ans["text"]) for ans in sample["answers"])
    em_scores.append(em)
    f1_scores.append(f1)
    results_json.append({
        "question": sample["question"],
        "truth": sample["answers"][0]["text"],
        "prediction": pred,
        "em": em, "f1": f1
    })

final_em = sum(em_scores) / len(em_scores) * 100
final_f1 = sum(f1_scores) / len(f1_scores) * 100

print(f"\nKẾT QUẢ CUỐI CÙNG:")
print(f"Exact Match (EM) : {final_em:.2f}%")
print(f"F1-score         : {final_f1:.2f}%")

with open("eval_results_batch.json", "w", encoding="utf-8") as f:
    json.dump({"em": final_em, "f1": final_f1, "time": end_time - start_time, "predictions": results_json}, f, ensure_ascii=False, indent=2)
